# Agentic Systems in Production

> Computational Analysis of Social Complexity
>
> Fall 2025, Spencer Lyon

**Prerequisites**

- All previous AI and Agentic Systems lectures (Weeks A1-A3)
- Agent-Based Models (Weeks 6-7)
- Game Theory and Networks (Weeks 3-5, 8-9)
- Understanding of system design and architecture concepts

**Outcomes**

- Design production-grade agent system architectures
- Implement monitoring, observability, and evaluation frameworks
- Apply reliability engineering patterns to AI agents
- Analyze real-world industry deployments and extract lessons
- Optimize for cost-performance trade-offs at scale
- Understand operational considerations for agent systems

**References**

- [GitHub Copilot Workspace Technical Report](https://github.blog/2024-04-29-github-copilot-workspace/)
- [Anthropic Claude Production Guide](https://docs.anthropic.com/en/docs/production)
- [OpenAI Production Best Practices](https://platform.openai.com/docs/guides/production-best-practices)
- [Google SRE Book](https://sre.google/sre-book/table-of-contents/) - Reliability engineering principles
- [AWS Well-Architected Framework](https://aws.amazon.com/architecture/well-architected/)

## From Prototype to Production

- Over the past weeks, we've built sophisticated AI agent systems
  - Single agents with tools and RAG (Week A1)
  - Multi-agent frameworks and type safety (Week A2)
  - Swarms, game theory, and digital twins (Week A3)
- All of these were prototypes - code that demonstrates concepts
- But now imagine you need to deploy this to real users
  - Thousands of requests per second
  - 99.9% uptime requirements
  - Cost constraints (API bills can exceed millions/month)
  - Security and compliance requirements
  - Regulatory scrutiny

This is the **production deployment** challenge.

### The Gap Between Research and Reality

Consider what happens when research code meets real users:

**In Research/Development**:
- "It works on my machine"
- Tolerate failures (just re-run)
- Debug interactively
- Optimize for development speed
- Single user (you)

**In Production**:
- Must work on all machines, all the time
- Failures cost money and trust
- Debug from logs and metrics
- Optimize for runtime performance and cost
- Thousands of concurrent users

The gap is enormous - and crossing it requires different skills.

### Why This Matters for Computational Social Science

You might think: "I'm a researcher, not a software engineer - why do I care?"

Three reasons:

**1. Reproducibility and Impact**
- Your agent-based models need to run reliably for other researchers
- Digital twins need to sync with real data continuously
- Tools you build should be usable beyond your laptop

**2. Real-World Applications**
- Policy simulations need to be trusted by decision-makers
- Economic models deployed by firms need 24/7 uptime
- Social network analysis tools scale to billions of interactions

**3. Understanding Deployed AI**
- AI agents are increasingly part of social and economic systems
- To study their impact, you need to understand how they actually work
- Production constraints shape agent behavior

**Example**: Remember Braess' Paradox from Week 8?
- In theory: adding capacity can worsen outcomes
- In production: adding servers can worsen latency (queue buildup)
- Same principle, different domain

Understanding production systems makes you a better computational social scientist.

## Production Architecture

### System Design Patterns

When deploying agent systems at scale, architecture matters enormously.

Let's explore the key patterns:

#### Pattern 1: Synchronous Request-Response

**Structure**:
```
User → API Server → Agent → LLM → Agent → API Server → User
```

**Characteristics**:
- User waits for complete response
- Simple to implement
- Easy to reason about

**Use When**:
- Response time < 30 seconds
- User expects immediate answer
- Example: Chatbot, Q&A system

**Challenges**:
- Timeout issues (LLM calls can be slow)
- Resource waste (server holds connection open)
- Doesn't scale to long-running tasks

#### Pattern 2: Asynchronous Task Queue

**Structure**:
```
User → API Server → Task Queue → Worker Pool → LLM
                ←  Task ID  ←
                
User → Check Status → Database → Worker writes here
```

**Characteristics**:
- User gets task ID immediately
- Poll for status or use webhooks
- Workers process independently

**Use When**:
- Tasks take > 30 seconds
- High request volume
- Example: Document analysis, batch processing

**Advantages**:
- Scales horizontally (add more workers)
- Handles backpressure gracefully
- Can prioritize tasks

#### Pattern 3: Event-Driven Architecture

**Structure**:
```
Event Source → Message Bus → Multiple Agents listening
                              ↓
                        Agent 1: Logs event
                        Agent 2: Updates dashboard
                        Agent 3: Triggers workflow
```

**Characteristics**:
- Agents react to events
- Decoupled components
- Publish-subscribe pattern

**Use When**:
- Multiple systems need to react to same data
- Real-time processing
- Example: Trading systems, social media monitoring

**Connection to Networks**: This is like information diffusion in social networks - events propagate through connected agents.

### Implementing Architecture Patterns in Julia

Let's build a simple async task queue system for agent work:

In [ ]:
import asyncio
from datetime import datetime
from uuid import uuid4
import pandas as pd
import numpy as np
from typing import Optional, Dict, Any, List
import time

In [ ]:
from enum import Enum
from dataclasses import dataclass, field

class TaskStatus(Enum):
    """Task status enum"""
    PENDING = "pending"
    RUNNING = "running"
    COMPLETED = "completed"
    FAILED = "failed"

@dataclass
class Task:
    """Represents a task in the system"""
    id: str
    status: TaskStatus
    created_at: datetime
    input: Dict[str, Any]
    started_at: Optional[datetime] = None
    completed_at: Optional[datetime] = None
    result: Optional[Any] = None
    error: Optional[str] = None
    retries: int = 0
    
    @classmethod
    def create(cls, input_data: Dict[str, Any]) -> 'Task':
        """Create a new task with generated ID"""
        return cls(
            id=str(uuid4()),
            status=TaskStatus.PENDING,
            created_at=datetime.now(),
            input=input_data
        )

@dataclass
class TaskQueue:
    """Task queue with persistence"""
    tasks: Dict[str, Task] = field(default_factory=dict)
    pending_queue: List[str] = field(default_factory=list)
    max_retries: int = 3
    
    def enqueue(self, task: Task) -> str:
        """Add a task to the queue"""
        self.tasks[task.id] = task
        self.pending_queue.append(task.id)
        return task.id
    
    def dequeue(self) -> Optional[Task]:
        """Remove and return the next task from queue"""
        if not self.pending_queue:
            return None
        task_id = self.pending_queue.pop(0)
        return self.tasks[task_id]
    
    def get_task(self, task_id: str) -> Optional[Task]:
        """Retrieve a task by ID"""
        return self.tasks.get(task_id)

print("Task queue infrastructure defined")

In [ ]:
import random

async def process_task(task: Task) -> Task:
    """
    Simulated worker that processes tasks.
    In production, this would call LLM APIs, run simulations, etc.
    Here we simulate with sleep and simple logic.
    """
    task.status = TaskStatus.RUNNING
    task.started_at = datetime.now()
    
    try:
        # Simulate processing time
        await asyncio.sleep(random.random() * 2)
        
        # Simulate 10% failure rate
        if random.random() < 0.1:
            raise Exception("Simulated LLM API timeout")
        
        # Successful processing
        processing_time = (datetime.now() - task.started_at).total_seconds()
        task.result = {
            "processed": True,
            "input_size": len(str(task.input)),
            "processing_time": processing_time
        }
        task.status = TaskStatus.COMPLETED
        task.completed_at = datetime.now()
        
    except Exception as e:
        task.status = TaskStatus.FAILED
        task.error = str(e)
        task.retries += 1
        task.completed_at = datetime.now()
    
    return task

async def worker_loop(queue: TaskQueue, worker_id: int, max_tasks: int = 10):
    """
    Worker that continuously processes tasks from queue.
    In production, this would run in a separate process/container.
    Here we simulate by processing a fixed number of tasks.
    """
    processed = 0
    
    while processed < max_tasks:
        task = queue.dequeue()
        
        if task is None:
            await asyncio.sleep(0.1)  # No tasks available
            continue
        
        print(f"Worker {worker_id} processing task {task.id}")
        await process_task(task)
        
        # Retry logic
        if task.status == TaskStatus.FAILED and task.retries < queue.max_retries:
            print(f"Worker {worker_id} retrying task {task.id} (attempt {task.retries + 1})")
            task.status = TaskStatus.PENDING
            queue.enqueue(task)
        
        processed += 1

print("Worker infrastructure defined")

### Testing the Task Queue

Let's simulate a production workload:

In [ ]:
# Create queue
queue = TaskQueue()

# Enqueue tasks
print("Enqueuing 20 tasks...")
task_ids = []
for i in range(20):
    task = Task.create({
        "type": "agent_query",
        "query": f"Analyze data point {i}",
        "priority": random.choice(["high", "medium", "low"])
    })
    task_id = queue.enqueue(task)
    task_ids.append(task_id)

print(f"Enqueued {len(task_ids)} tasks")
print(f"Pending: {len(queue.pending_queue)}")

In [ ]:
# Simulate multiple workers processing in parallel
print("\nStarting 3 workers...")

# Run workers concurrently using asyncio.gather
await asyncio.gather(
    worker_loop(queue, 1, max_tasks=7),
    worker_loop(queue, 2, max_tasks=7),
    worker_loop(queue, 3, max_tasks=7)
)

print("\nAll workers finished")

In [ ]:
# Analyze results
completed = sum(1 for t in queue.tasks.values() if t.status == TaskStatus.COMPLETED)
failed = sum(1 for t in queue.tasks.values() if t.status == TaskStatus.FAILED)
total_retries = sum(t.retries for t in queue.tasks.values())

print("\n=== Results ===")
print(f"Completed: {completed}")
print(f"Failed: {failed}")
print(f"Total retries: {total_retries}")

# Calculate processing times
processing_times = []
for task in queue.tasks.values():
    if task.status == TaskStatus.COMPLETED and task.started_at and task.completed_at:
        duration = (task.completed_at - task.started_at).total_seconds()
        processing_times.append(duration)

if processing_times:
    print("\nProcessing times:")
    print(f"  Mean: {np.mean(processing_times):.2f}s")
    print(f"  P50: {np.median(processing_times):.2f}s")
    print(f"  P95: {np.percentile(processing_times, 95):.2f}s")
    print(f"  Max: {np.max(processing_times):.2f}s")

### Key Insights from Architecture

What we've seen:

**1. Decoupling is Critical**
- Queue separates request acceptance from processing
- Can scale workers independently of API servers
- Failed workers don't block new requests

**2. Retry Logic Matters**
- LLM APIs fail ~1-5% of the time (timeouts, rate limits, etc.)
- Automatic retries dramatically improve reliability
- Exponential backoff prevents thundering herd

**3. Observability is Essential**
- We tracked: status, retries, processing times
- In production: also track costs, errors, user satisfaction
- Metrics enable optimization

**Connection to Game Theory**: This is a coordination game!
- Workers compete for tasks (avoid starvation)
- Need to avoid duplicate work (waste)
- Queue is a coordination mechanism

## State Management and Persistence

### The Conversation State Challenge

Remember multi-agent conversations from Week A1?
- Each message builds on previous context
- Agents maintain state across turns
- But what happens when:
  - Server restarts?
  - User closes browser and comes back tomorrow?
  - Request gets routed to different server?

You need **persistent state management**.

### State Storage Options

**In-Memory (Not Persistent)**:
```julia
conversation_state = Dict{String, Vector{Message}}()
```

**Pros**: Fast (microseconds)
**Cons**: Lost on restart, doesn't scale across servers
**Use**: Development, caching

**Database (Persistent)**:
```julia
# Store in PostgreSQL, MongoDB, etc.
save_conversation(db, user_id, messages)
```

**Pros**: Reliable, queryable, scales
**Cons**: Slower (milliseconds), costs money
**Use**: Production long-term storage

**Cache + Database (Hybrid)**:
```julia
# Check cache first
state = get_from_cache(redis, conversation_id)
if state === nothing
    state = load_from_db(db, conversation_id)
    put_in_cache(redis, conversation_id, state)
end
```

**Pros**: Fast + reliable
**Cons**: More complex
**Use**: Production at scale

### State Size Growth

A critical production issue: conversation history grows unbounded!

**The Problem**:
- Each message adds ~500-2000 tokens
- LLMs have context limits (100K-200K tokens)
- Costs scale with context size

**Solutions**:

**1. Sliding Window**:
```julia
function get_context(messages, window_size=10)
    return messages[max(1, end-window_size+1):end]
end
```
Keep only recent messages.

**2. Summarization**:
```julia
function compress_history(messages, threshold=20)
    if length(messages) > threshold
        old_messages = messages[1:threshold]
        summary = llm_summarize(old_messages)
        return [summary, messages[threshold+1:end]...]
    end
    return messages
end
```
Compress old messages into summary.

**3. Retrieval-Augmented Memory** (from Week A1):
- Store full history in vector database
- Retrieve only relevant past messages
- Keep context size bounded

**Connection to ABMs**: Like our money model from Week 7!
- Agents had finite wealth (bounded state)
- Exchange rules kept system stable
- Here: bounded context, compression rules maintain stability

## Reliability Engineering

### Failure Modes in Agent Systems

LLM-based agents fail in unique ways. Let's catalog them:

**1. API Failures**
- Timeouts (request takes too long)
- Rate limits (too many requests)
- 500 errors (provider infrastructure issues)
- 429 errors (quota exceeded)

**Frequency**: 1-5% of requests
**Impact**: High (user sees error)
**Mitigation**: Retries with exponential backoff

**2. Output Format Errors**
- Asked for JSON, got text
- JSON missing required fields
- Hallucinated function names

**Frequency**: 0.5-2% (depends on prompt quality)
**Impact**: Medium (can retry with better prompt)
**Mitigation**: Structured output APIs, validation, retry with clarification

**3. Semantic Failures**
- Correct format, wrong content
- Hallucinated facts
- Missed key information

**Frequency**: Varies wildly (5-30% depending on task)
**Impact**: High (produces incorrect results)
**Mitigation**: Multi-agent verification, human-in-loop, confidence scoring

**4. Cost Blowups**
- Agent gets stuck in loop
- Generates massive context
- Uses expensive model unnecessarily

**Frequency**: Rare (<0.1%) but catastrophic
**Impact**: Critical (thousands of dollars in minutes)
**Mitigation**: Circuit breakers, token limits, model selection

**5. Latency Issues**
- Request takes > 30 seconds
- User abandons (high bounce rate)
- Cascading delays

**Frequency**: 5-15% of requests (depends on model, context size)
**Impact**: Medium (degrades UX)
**Mitigation**: Streaming responses, async patterns, model optimization

### Pattern: Circuit Breaker

A critical reliability pattern from systems engineering.

**The Problem**:
- LLM provider has an outage
- Your system keeps sending requests
- All requests fail and retry
- Amplifies the problem (thundering herd)

**The Solution: Circuit Breaker**

Like an electrical circuit breaker, it "opens" when failures exceed a threshold:

**States**:
1. **CLOSED**: Normal operation, requests go through
2. **OPEN**: Too many failures, block all requests immediately
3. **HALF_OPEN**: Testing if service recovered, allow few requests

**Transitions**:
- CLOSED → OPEN: When failure rate > threshold (e.g., 50% over 10 requests)
- OPEN → HALF_OPEN: After timeout (e.g., 60 seconds)
- HALF_OPEN → CLOSED: If test requests succeed
- HALF_OPEN → OPEN: If test requests fail

Let's implement it:

In [ ]:
from enum import Enum
from typing import Callable, TypeVar

class CircuitState(Enum):
    """Circuit breaker states"""
    CLOSED = "closed"
    OPEN = "open"
    HALF_OPEN = "half_open"

T = TypeVar('T')

@dataclass
class CircuitBreaker:
    """Circuit breaker for handling failures"""
    state: CircuitState = CircuitState.CLOSED
    failure_count: int = 0
    success_count: int = 0
    last_failure_time: Optional[datetime] = None
    
    # Configuration
    failure_threshold: int = 5  # Open circuit after this many failures
    success_threshold: int = 2  # Close circuit after this many successes
    timeout_seconds: int = 60   # Time to wait before trying again
    
    def call_with_breaker(self, fn: Callable[[], T]) -> T:
        """Execute a function with circuit breaker protection"""
        # Check if circuit is open
        if self.state == CircuitState.OPEN:
            # Check if timeout expired
            if self.last_failure_time and \
               (datetime.now() - self.last_failure_time).total_seconds() > self.timeout_seconds:
                self.state = CircuitState.HALF_OPEN
                self.success_count = 0
                print("Circuit breaker: OPEN → HALF_OPEN (testing recovery)")
            else:
                raise Exception("Circuit breaker is OPEN - service unavailable")
        
        # Try to call function
        try:
            result = fn()
            
            # Success!
            self.failure_count = 0
            self.success_count += 1
            
            # If in HALF_OPEN, check if we can close
            if self.state == CircuitState.HALF_OPEN and self.success_count >= self.success_threshold:
                self.state = CircuitState.CLOSED
                print("Circuit breaker: HALF_OPEN → CLOSED (service recovered)")
            
            return result
            
        except Exception as e:
            # Failure
            self.failure_count += 1
            self.success_count = 0
            self.last_failure_time = datetime.now()
            
            # Check if we should open circuit
            if self.failure_count >= self.failure_threshold:
                old_state = self.state
                self.state = CircuitState.OPEN
                print(f"Circuit breaker: {old_state.value} → OPEN (too many failures)")
            
            raise e

print("Circuit breaker implementation complete")

### Testing the Circuit Breaker

Let's simulate an LLM API that's flaky:

In [ ]:
# Simulate flaky LLM API
call_count = [0]

def flaky_llm_call():
    """Simulated API that fails first 10 times, then succeeds"""
    call_count[0] += 1
    
    # First 10 calls fail (simulating outage)
    # Then succeed (simulating recovery)
    if call_count[0] <= 10:
        raise Exception("API timeout")
    
    return "Success!"

# Create circuit breaker
cb = CircuitBreaker(failure_threshold=3, success_threshold=2, timeout_seconds=1)

# Try to call API 20 times
print("Testing circuit breaker with flaky API...\n")

for i in range(1, 21):
    try:
        result = cb.call_with_breaker(flaky_llm_call)
        print(f"Call {i}: SUCCESS - {result}")
    except Exception as e:
        print(f"Call {i}: FAILED - {type(e).__name__}")
    
    time.sleep(0.3)  # Small delay

print(f"\nTotal actual API calls made: {call_count[0]}")
print(f"Calls blocked by circuit breaker: {20 - call_count[0]}")

### Circuit Breaker Insights

What happened:

1. First 3 calls failed → circuit opened
2. Several calls immediately rejected (didn't even try API)
3. After timeout, entered HALF_OPEN
4. Test calls eventually succeeded → circuit closed
5. Remaining calls succeeded

**Benefits**:
- Prevents cascading failures
- Reduces load on failing service
- Fails fast (no waiting for timeouts)
- Automatic recovery testing

**Production Use**:
- Apply to all external API calls
- Use different breakers for different services
- Monitor breaker state (alerts when open)
- Configure thresholds based on SLAs

**Connection to Game Theory**: This is a reputation system!
- API builds reputation through success
- Loses reputation through failures
- Circuit breaker is the enforcement mechanism

## Monitoring and Observability

### The Three Pillars of Observability

You can't improve what you can't measure. Production systems need:

**1. Metrics** (Quantitative Measurements)
- Request rate (requests/second)
- Error rate (errors/total requests)
- Latency (P50, P95, P99)
- Cost ($/request, $/day)
- Token usage (input/output tokens)

**2. Logs** (Event Records)
- Structured logs (JSON) with context
- Request ID for tracing
- Errors with stack traces
- Agent decision logs

**3. Traces** (Request Flow)
- Follow a request through system
- See all steps and timing
- Identify bottlenecks

### Key Metrics for Agent Systems

Beyond standard metrics, agent systems need:

**Agent-Specific Metrics**:
- Tool call accuracy (% of valid calls)
- Reasoning steps per task
- Context size over time
- Multi-agent coordination time
- Verification/validation pass rate

**Business Metrics**:
- Task completion rate
- User satisfaction (thumbs up/down)
- Time to value
- Human intervention rate

**Cost Metrics**:
- Cost per task
- Cost per user
- Cost per outcome (e.g., cost per successful analysis)
- Model tier usage (expensive vs cheap models)

### Implementing Metrics Collection

Let's build a simple metrics system:

In [ ]:
@dataclass
class MetricsCollector:
    """Metrics collector for agent systems"""
    # Counters
    total_requests: int = 0
    successful_requests: int = 0
    failed_requests: int = 0
    
    # Latency tracking (in seconds)
    latencies: List[float] = field(default_factory=list)
    
    # Cost tracking (in dollars)
    total_cost: float = 0.0
    costs_by_model: Dict[str, float] = field(default_factory=dict)
    
    # Token tracking
    total_input_tokens: int = 0
    total_output_tokens: int = 0
    
    # Agent-specific
    tool_calls: int = 0
    invalid_tool_calls: int = 0
    reasoning_steps: List[int] = field(default_factory=list)
    
    def record_request(self, success: bool, latency: float, 
                      input_tokens: int, output_tokens: int,
                      model: str = "gpt-4", tool_calls: int = 0, 
                      invalid_tools: int = 0, reasoning_steps: int = 1):
        """Record metrics for a request"""
        self.total_requests += 1
        
        if success:
            self.successful_requests += 1
        else:
            self.failed_requests += 1
        
        self.latencies.append(latency)
        
        # Calculate cost (simplified pricing)
        if model == "gpt-4":
            cost = input_tokens * 0.00003 + output_tokens * 0.00006
        else:  # gpt-3.5
            cost = input_tokens * 0.000001 + output_tokens * 0.000002
        
        self.total_cost += cost
        self.costs_by_model[model] = self.costs_by_model.get(model, 0.0) + cost
        
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        
        self.tool_calls += tool_calls
        self.invalid_tool_calls += invalid_tools
        self.reasoning_steps.append(reasoning_steps)
    
    def print_metrics(self):
        """Print comprehensive metrics report"""
        print("\n=== METRICS REPORT ===")
        
        print("\nRequests:")
        print(f"  Total: {self.total_requests}")
        print(f"  Successful: {self.successful_requests} ({100 * self.successful_requests / self.total_requests:.1f}%)")
        print(f"  Failed: {self.failed_requests} ({100 * self.failed_requests / self.total_requests:.1f}%)")
        
        if self.latencies:
            print("\nLatency:")
            print(f"  P50: {np.percentile(self.latencies, 50):.2f}s")
            print(f"  P95: {np.percentile(self.latencies, 95):.2f}s")
            print(f"  P99: {np.percentile(self.latencies, 99):.2f}s")
            print(f"  Max: {np.max(self.latencies):.2f}s")
        
        print("\nCost:")
        print(f"  Total: ${self.total_cost:.4f}")
        print(f"  Per request: ${self.total_cost / self.total_requests:.6f}")
        for model, cost in self.costs_by_model.items():
            print(f"  {model}: ${cost:.4f}")
        
        print("\nTokens:")
        print(f"  Input: {self.total_input_tokens}")
        print(f"  Output: {self.total_output_tokens}")
        print(f"  Total: {self.total_input_tokens + self.total_output_tokens}")
        print(f"  Avg input per request: {self.total_input_tokens / self.total_requests:.0f}")
        
        print("\nAgent Performance:")
        print(f"  Tool calls: {self.tool_calls}")
        if self.tool_calls > 0:
            print(f"  Invalid tool calls: {self.invalid_tool_calls} ({100 * self.invalid_tool_calls / max(1, self.tool_calls):.1f}%)")
        if self.reasoning_steps:
            print(f"  Avg reasoning steps: {np.mean(self.reasoning_steps):.1f}")

print("Metrics system defined")

### Simulating Production Traffic

Let's generate realistic metrics:

In [ ]:
metrics = MetricsCollector()

# Simulate 1000 requests with varying characteristics
print("Simulating 1000 production requests...")

for i in range(1000):
    # Most requests succeed
    success = random.random() > 0.05
    
    # Latency varies (log-normal distribution)
    base_latency = random.random() * 3 + 0.5
    latency = base_latency if success else base_latency * 2
    
    # Token counts
    input_tokens = random.randint(500, 3000)
    output_tokens = random.randint(100, 1000)
    
    # 30% use GPT-4, rest use GPT-3.5
    model = "gpt-4" if random.random() < 0.3 else "gpt-3.5"
    
    # Tool usage
    tool_calls = random.randint(0, 5)
    invalid_tools = 1 if random.random() < 0.02 else 0  # 2% invalid
    
    # Reasoning steps
    reasoning_steps = random.randint(1, 8)
    
    metrics.record_request(
        success, latency,
        input_tokens, output_tokens,
        model, tool_calls, invalid_tools,
        reasoning_steps
    )

metrics.print_metrics()

### Using Metrics for Optimization

Metrics tell you where to optimize:

**If P99 latency is high** (> 10s):
- Use streaming responses
- Implement async patterns
- Cache common queries
- Use smaller/faster models

**If cost is high** (> $0.01/request):
- Use cheaper models (GPT-3.5 vs GPT-4)
- Reduce context size
- Cache results
- Batch requests

**If error rate is high** (> 5%):
- Improve prompts
- Add retry logic
- Implement circuit breakers
- Add validation

**If invalid tool calls are high** (> 5%):
- Simplify tool descriptions
- Add examples to prompts
- Use structured output APIs
- Reduce number of tools

**Production Rule**: Instrument everything, analyze regularly, optimize iteratively.

## Case Studies from Industry

### Case Study 1: GitHub Copilot Workspace

**System**: AI agent that writes code from natural language descriptions

**Scale**:
- Millions of users
- Billions of completions
- Real-time latency requirements

**Architecture**:
```
User IDE → Edge Cache → Load Balancer → Agent Cluster
                                             ↓
                                    Multiple LLM Providers
                                    (OpenAI, Anthropic, etc.)
```

**Key Decisions**:

**1. Multi-Provider Strategy**
- Use multiple LLM providers
- Route based on availability and cost
- Fallback if primary fails
- **Lesson**: Don't depend on single provider

**2. Aggressive Caching**
- Cache common code completions
- Semantic similarity for cache hits
- Reduces cost by 40%
- **Lesson**: Caching is critical at scale

**3. Streaming Responses**
- Show tokens as they're generated
- User sees progress immediately
- Can cancel slow completions
- **Lesson**: Perceived latency matters as much as actual latency

**4. Model Selection by Task**
- Simple completions: Small fast model
- Complex refactoring: Larger model
- Automatic routing
- **Lesson**: One size doesn't fit all

**5. Telemetry for Quality**
- Track acceptance rate (% of suggestions used)
- A/B test prompts and models
- Continuous improvement loop
- **Lesson**: Measure outcome quality, not just system metrics

**Results**:
- 99.9% uptime
- <100ms P50 latency
- Cost reduced 3x through optimization
- 55% of code accepted (very high for AI suggestions)

**Connection to Course**: This is a digital twin of developer behavior!
- Agent learns what completions developers accept
- Adapts suggestions based on context
- Continuous feedback loop (Week A3 digital twins)

### Case Study 2: Intercom AI Customer Service Agent

**System**: AI chatbot answering customer support questions

**Scale**:
- 10,000+ companies using it
- Millions of conversations/month
- 24/7 availability requirement

**Challenge**: Balance automation with quality
- Incorrect answers damage brand
- But humans are expensive
- Need to know when to escalate

**Architecture**:
```
Customer → AI Agent → {Confidence Check}
                           ↓
                    High: Send answer
                    Medium: Show answer + "Was this helpful?"
                    Low: Escalate to human
```

**Key Decisions**:

**1. Confidence Scoring**
- Every response has confidence score
- Based on: retrieval quality, LLM uncertainty, past accuracy
- Different actions by confidence level
- **Lesson**: AI should know what it doesn't know

**2. Human-in-Loop by Design**
- AI drafts response
- Human can edit before sending
- Human edits become training data
- **Lesson**: Augment humans, don't replace

**3. RAG with Company Knowledge Base**
- Each company has custom docs
- Embed and index on setup
- Retrieve relevant docs for each query
- **Lesson**: Domain knowledge is critical (Week A1 RAG)

**4. Fallback Chains**
- Try to answer directly
- If fails, search knowledge base
- If fails, suggest related articles
- If fails, escalate to human
- **Lesson**: Graceful degradation beats hard failures

**5. Continuous Evaluation**
- Users rate responses (thumbs up/down)
- Track resolution rate
- Monitor escalation rate
- **Lesson**: User feedback is ground truth

**Results**:
- 70% of queries fully automated
- 85% user satisfaction (vs 80% with humans)
- 30% cost reduction
- Average response time: 15 seconds (vs 2 hours with humans)

**Connection to Game Theory**: This is a signaling game!
- AI signals confidence
- User decides whether to trust
- Calibration is key (don't over-promise)

### Case Study 3: Financial Trading Agent (Anonymous Hedge Fund)

**System**: AI agent analyzing news and making trading recommendations

**Scale**:
- Real money at stake (millions of dollars)
- Millisecond latency requirements
- Regulatory compliance critical

**Architecture**:
```
News Feed → Content Filter → Sentiment Agent → Risk Agent → Human Trader
                                     ↓              ↓              ↓
                               Trading Signal → Risk Check → Execute
```

**Key Decisions**:

**1. Multi-Agent Verification**
- Sentiment agent analyzes news
- Risk agent checks portfolio impact
- Compliance agent checks regulations
- All must agree before trading
- **Lesson**: Critical decisions need multiple perspectives (Week A3 swarms)

**2. Hybrid AI-Human**
- AI generates signals
- Human makes final decision
- But human can set "auto-execute" rules
- **Lesson**: Gradually increase automation based on trust

**3. Extensive Backtesting**
- Test on historical data
- Simulate agent decisions
- Measure hypothetical returns
- **Lesson**: Digital twin for testing before real deployment (Week A3)

**4. Explainability Requirements**
- Every recommendation has explanation
- Show which news articles influenced decision
- Audit trail for regulators
- **Lesson**: Black boxes don't work in regulated industries

**5. Adversarial Testing**
- Red team tries to manipulate agent
- Test with fake news, pump-and-dump schemes
- Measure robustness
- **Lesson**: Assume adversarial environment (game theory)

**Results**:
- 15% improvement in Sharpe ratio
- 50% reduction in "false positive" trades
- Zero regulatory violations
- But: only handles 30% of trades (rest too complex)

**Connection to Course**:
- Game theory: Strategic interaction with other traders (Week 8-9)
- Networks: Information flow through news networks (Week 3-5)
- ABMs: Simulating market with AI trader agents (Week 6-7)

### Common Lessons Across Case Studies

What do these very different deployments share?

**1. Multi-Provider/Multi-Model is Standard**
- Don't depend on single LLM provider
- Use different models for different tasks
- Have fallback options

**2. Confidence/Uncertainty Estimation is Critical**
- AI should know what it doesn't know
- Different actions based on confidence
- Escalation paths for uncertain cases

**3. Human-in-Loop is Universal**
- Even "autonomous" agents have human oversight
- Gradually increase automation based on trust
- Humans handle edge cases

**4. Continuous Evaluation Drives Improvement**
- Measure outcome quality, not just system metrics
- User feedback is ground truth
- A/B test everything

**5. Domain Knowledge Makes or Breaks**
- RAG with company-specific knowledge
- Fine-tuning on domain data
- Custom tooling for domain tasks

**6. Cost Optimization is Ongoing**
- Start with expensive models, optimize down
- Caching is critical
- Model routing by task complexity

**7. Reliability Patterns are Non-Negotiable**
- Retries, circuit breakers, fallbacks
- Graceful degradation
- Comprehensive monitoring

## Performance Optimization

### The Optimization Hierarchy

Optimize in this order (biggest impact first):

**1. Reduce Requests** (10-100x improvement)
- Cache results
- Batch operations
- Deduplicate queries

**2. Use Cheaper Models** (3-20x improvement)
- GPT-4 → GPT-3.5: 20x cheaper
- Claude Opus → Sonnet: 3x cheaper
- Route by task complexity

**3. Reduce Context Size** (2-5x improvement)
- Summarize old messages
- Remove unnecessary examples
- Compress system prompts

**4. Parallel Execution** (2-3x improvement)
- Multiple independent tool calls
- Parallel agent reasoning
- Batch processing

**5. Code-Level Optimization** (1.1-1.5x improvement)
- Async I/O
- Connection pooling
- Efficient data structures

Let's explore each:

### Optimization 1: Semantic Caching

**The Idea**: Many user queries are similar
- "What's the weather in NYC?" ≈ "Tell me NYC weather"
- Don't call LLM for duplicate queries
- Use embedding similarity for fuzzy matching

In [ ]:
@dataclass
class SemanticCache:
    """Simple semantic cache using embeddings"""
    entries: List[Dict[str, Any]] = field(default_factory=list)
    max_size: int = 1000
    similarity_threshold: float = 0.95
    hits: int = 0
    misses: int = 0

def simple_embedding(text: str) -> List[float]:
    """
    Simplified embedding for demo purposes.
    In production, use actual embedding model (OpenAI embeddings, sentence-transformers).
    """
    # Hash-based fake embedding for demo
    h = hash(text.lower().strip())
    return [(((h >> (8*i)) & 0xFF) / 255.0) for i in range(8)]

def cosine_similarity(a: List[float], b: List[float]) -> float:
    """Calculate cosine similarity between two vectors"""
    dot_product = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(y * y for y in b) ** 0.5
    return dot_product / (norm_a * norm_b) if norm_a and norm_b else 0.0

def cache_get(cache: SemanticCache, query: str) -> Optional[str]:
    """Retrieve from cache if similar query exists"""
    query_emb = simple_embedding(query)
    
    best_match = None
    best_similarity = 0.0
    
    for entry in cache.entries:
        similarity = cosine_similarity(query_emb, entry["embedding"])
        if similarity > best_similarity and similarity >= cache.similarity_threshold:
            best_similarity = similarity
            best_match = entry
    
    if best_match:
        cache.hits += 1
        return best_match["response"]
    else:
        cache.misses += 1
        return None

def cache_put(cache: SemanticCache, query: str, response: str):
    """Add a query-response pair to cache"""
    query_emb = simple_embedding(query)
    
    entry = {
        "query": query,
        "embedding": query_emb,
        "response": response,
        "timestamp": datetime.now()
    }
    
    cache.entries.append(entry)
    
    # Evict oldest if over max size
    if len(cache.entries) > cache.max_size:
        cache.entries.pop(0)

def cache_hit_rate(cache: SemanticCache) -> float:
    """Calculate cache hit rate"""
    total = cache.hits + cache.misses
    return cache.hits / total if total > 0 else 0.0

print("Semantic cache implementation complete")

### Testing Semantic Cache

Let's simulate queries with some duplication:

In [ ]:
cache = SemanticCache()

# Simulate 100 queries with some duplication
queries = [
    "What's the weather like?",
    "Tell me about the weather",
    "How's the weather today?",
    "Explain quantum mechanics",
    "What is quantum mechanics?",
    "Quantum mechanics explanation",
    "Calculate 123 + 456",
    "What is 123 plus 456?",
    "Add 123 and 456"
]

# Simulate 100 requests with repetition
api_calls = 0

for i in range(100):
    query = random.choice(queries)
    
    # Check cache
    cached_response = cache_get(cache, query)
    
    if cached_response is not None:
        # Cache hit! No API call needed
        response = cached_response
    else:
        # Cache miss - need to call API
        api_calls += 1
        response = f"Simulated LLM response for: {query}"
        cache_put(cache, query, response)

hit_rate = cache_hit_rate(cache)

print("\n=== CACHE PERFORMANCE ===")
print(f"Total requests: 100")
print(f"Cache hits: {cache.hits}")
print(f"Cache misses: {cache.misses}")
print(f"Hit rate: {100 * hit_rate:.1f}%")
print(f"API calls made: {api_calls}")
print(f"API calls saved: {100 - api_calls}")
print(f"Cost reduction: {100 * (100 - api_calls) / 100:.1f}%")

### Caching Insights

**Impact**:
- 40-60% cache hit rate is typical
- Translates to 40-60% cost reduction
- Also 40-60% latency reduction (cache is instant)

**Production Considerations**:

**1. Embedding Quality Matters**
- Use real embeddings (OpenAI embeddings API, sentence-transformers)
- Better embeddings → better similarity matching

**2. Tune Similarity Threshold**
- Too high (>0.98): Few hits, many misses
- Too low (<0.90): False hits, wrong answers
- Sweet spot: ~0.95

**3. Invalidation is Hard**
- Time-based expiry (evict after N hours)
- Manual invalidation (when data changes)
- LRU eviction (least recently used)

**4. Multi-Level Caching**
- L1: In-memory (instant)
- L2: Redis (milliseconds)
- L3: Database (tens of milliseconds)

**Connection to Networks**: Cache is a hub in information network!
- Queries flow through cache
- Cache aggregates and serves
- Reduces load on downstream services

### Optimization 2: Intelligent Model Selection

Not all tasks need GPT-4. Route intelligently:

**Simple Tasks** → Cheap Model (GPT-3.5, Claude Haiku)
- Summarization
- Classification
- Simple Q&A
- Data extraction

**Complex Tasks** → Expensive Model (GPT-4, Claude Opus)
- Multi-step reasoning
- Creative generation
- Code generation
- Strategic planning

**Cost Comparison**:
```
GPT-4: $30 / 1M input tokens
GPT-3.5: $1.50 / 1M input tokens (20x cheaper)

Claude Opus: $15 / 1M input tokens
Claude Sonnet: $3 / 1M input tokens (5x cheaper)
Claude Haiku: $0.80 / 1M input tokens (19x cheaper)
```

**Strategy**: Start with cheap model, escalate if needed

```julia
function intelligent_call(query, context)
    # First, classify complexity
    complexity = estimate_complexity(query)
    
    if complexity < 0.3
        model = "gpt-3.5-turbo"
    elseif complexity < 0.7
        model = "claude-sonnet"
    else
        model = "gpt-4"
    end
    
    response = call_llm(model, query, context)
    
    # Validate quality
    if quality_check(response) < 0.8 && model != "gpt-4"
        # Escalate to better model
        response = call_llm("gpt-4", query, context)
    end
    
    return response
end
```

**Result**: 3-5x cost reduction with minimal quality loss

## Exercises

### Exercise 1: Design a Production Architecture

**Scenario**: You're deploying a research paper analysis agent for academics.

**Requirements**:
- Users upload PDFs (1-50 pages)
- Agent extracts: methods, findings, limitations
- Generates structured summary
- Compares to related papers
- Expected load: 1000 papers/day

**Tasks**:
1. Design the system architecture
   - Synchronous or asynchronous?
   - How do you handle long PDFs?
   - Where does state live?

2. Identify failure modes
   - What can go wrong?
   - How do you detect failures?
   - What's your recovery strategy?

3. Plan for scale
   - What if load increases 10x?
   - What are your bottlenecks?
   - How do you scale horizontally?

4. Cost estimation
   - Estimate tokens per paper
   - Calculate daily cost
   - Identify optimization opportunities

**Deliverable**: Architecture diagram + written analysis (1-2 pages)

### Exercise 2: Implement Rate Limiting

**Objective**: Protect your system from abuse and manage costs.

**Requirements**:
- Limit each user to 100 requests/hour
- Different limits for different tiers (free, pro, enterprise)
- Return informative error when limit exceeded
- Track usage for billing

**Implementation**:

1. Create a `RateLimiter` struct with:
   - Token bucket algorithm
   - Per-user tracking
   - Configurable limits

2. Add to request pipeline:
   ```julia
   function handle_request(limiter, user_id, request)
       if !check_rate_limit(limiter, user_id)
           return error_response("Rate limit exceeded")
       end
       return process_request(request)
   end
   ```

3. Test with simulated traffic

**Challenge**: Implement a "burst" allowance - users can exceed limit briefly but must recover.

**Connection to Game Theory**: This is mechanism design!
- How do you prevent abuse?
- What incentives do limits create?
- How do users game the system?

In [ ]:
# TODO: Your implementation here

### Exercise 3: Build an Evaluation Framework

**Objective**: Systematically measure agent quality.

**Scenario**: Network analysis agent (from Week 3-5)

**Tasks**:

1. **Create test suite**:
   - 20 questions about network properties
   - Known correct answers
   - Range of difficulties

2. **Define metrics**:
   - Accuracy (% correct answers)
   - Reasoning quality (1-5 scale)
   - Latency (time to answer)
   - Cost (tokens used)

3. **Implement evaluation**:
   ```julia
   function evaluate_agent(agent, test_suite)
       results = []
       for test in test_suite
           response = agent(test.question)
           score = grade_response(response, test.answer)
           push!(results, score)
       end
       return results
   end
   ```

4. **A/B test prompts**:
   - Test 3 different system prompts
   - Compare metrics
   - Choose winner

**Deliverable**: Evaluation framework + results comparing prompts

In [ ]:
# TODO: Your implementation here

### Exercise 4: Optimize for Cost

**Scenario**: Your agent system costs $10,000/month. Reduce to $5,000 without sacrificing quality.

**Current System**:
- 100,000 requests/month
- Average: 3000 input tokens, 800 output tokens
- All using GPT-4
- No caching
- 30% of queries are simple classifications

**Optimization Strategies to Implement**:

1. **Add caching**
   - Assume 40% hit rate
   - Calculate savings

2. **Route simple queries to GPT-3.5**
   - 30% can use cheaper model
   - Calculate savings

3. **Compress system prompt**
   - Current: 500 tokens
   - Optimized: 200 tokens
   - Calculate savings

4. **Batch similar requests**
   - Process multiple queries in one call
   - Saves on repeated context
   - Calculate savings

**Deliverable**: Cost breakdown before/after each optimization

In [ ]:
# Cost calculation helper
def calculate_monthly_cost(requests: int, input_tokens: int, output_tokens: int, model: str = "gpt-4") -> float:
    """Calculate monthly cost for given usage"""
    if model == "gpt-4":
        input_cost = 0.03 / 1000  # $30 per 1M tokens
        output_cost = 0.06 / 1000
    else:  # gpt-3.5
        input_cost = 0.0015 / 1000
        output_cost = 0.002 / 1000
    
    total = requests * (input_tokens * input_cost + output_tokens * output_cost)
    return total

# TODO: Calculate baseline cost
baseline_cost = calculate_monthly_cost(100_000, 3000, 800, "gpt-4")
print(f"Baseline cost: ${baseline_cost:.2f}")

# TODO: Calculate cost with each optimization
# TODO: Show cumulative savings

### Exercise 5: Production Deployment Plan

**Objective**: Create a deployment plan for your course project.

**Requirements**:

1. **Architecture Document**
   - System diagram
   - Component responsibilities
   - Data flow
   - External dependencies

2. **Reliability Plan**
   - Failure modes and mitigations
   - Retry strategies
   - Circuit breakers
   - Fallback behavior

3. **Monitoring Plan**
   - Key metrics to track
   - Alert thresholds
   - Dashboard design
   - Incident response

4. **Cost Model**
   - Token usage estimates
   - Monthly cost projections
   - Optimization strategies
   - Scaling considerations

5. **Testing Plan**
   - Test suite design
   - Evaluation metrics
   - A/B testing strategy
   - Quality gates

**Deliverable**: 3-5 page deployment plan

**Purpose**: This is what you'd actually present to stakeholders before deployment.

## Summary

In this lecture, we've explored production deployment of agentic AI systems:

✓ **Production architectures**: Synchronous, asynchronous, event-driven patterns

✓ **State management**: Persistence, caching, context compression

✓ **Reliability engineering**: Circuit breakers, retry logic, failure modes

✓ **Monitoring and observability**: Metrics, logs, traces

✓ **Real-world case studies**: GitHub Copilot, Intercom, financial trading

✓ **Performance optimization**: Caching, model selection, cost reduction

**Key Takeaways**:

1. **Production is fundamentally different from research**
   - Reliability, cost, scale matter
   - Need comprehensive monitoring
   - Continuous optimization required

2. **Architecture decisions have huge impact**
   - Async vs sync changes everything
   - State management is critical
   - Decoupling enables scaling

3. **Reliability requires multiple patterns**
   - Retries, circuit breakers, fallbacks
   - Graceful degradation
   - Human-in-loop as final safety net

4. **Optimization has clear priorities**
   - Reduce requests (caching) → 10-100x
   - Cheaper models → 3-20x
   - Context compression → 2-5x
   - Parallelism → 2-3x

5. **Real deployments share common patterns**
   - Multi-provider/multi-model
   - Confidence-based routing
   - Human-in-loop
   - Continuous evaluation

**Connections to Course Themes**:

- **Networks** (Week 3-5): System architecture as network, caching as hub
- **Game Theory** (Week 8-9): Rate limiting is mechanism design, circuit breakers are reputation systems
- **ABMs** (Week 6-7): State management like bounded agent states, emergence in distributed systems
- **Digital Twins** (Week A3): Production monitoring enables continuous calibration

**The Big Picture**:

We've come full circle:
- Started with simple rule-based agents (Schelling model)
- Learned about AI agents and LLMs
- Built multi-agent systems and swarms
- Applied game theory and networks to agent interactions
- Now: deploying these systems in the real world

Production deployment isn't just engineering - it's applied computational social science:
- Understanding how systems scale
- Analyzing trade-offs and constraints
- Designing mechanisms for reliability
- Measuring and optimizing outcomes

**Next Lecture**: We'll explore the ethical and safety considerations of deployed agent systems - bias, fairness, accountability, and societal impact.

## Further Reading

### Production Systems
- [Google SRE Book](https://sre.google/sre-book/table-of-contents/) - Comprehensive reliability engineering
- [AWS Well-Architected Framework](https://aws.amazon.com/architecture/well-architected/) - System design principles
- [Designing Data-Intensive Applications](https://dataintensive.net/) - Martin Kleppmann's essential book

### AI in Production
- [Anthropic Production Guide](https://docs.anthropic.com/en/docs/production)
- [OpenAI Best Practices](https://platform.openai.com/docs/guides/production-best-practices)
- [LangSmith for Monitoring](https://docs.smith.langchain.com/) - Observability for LLM apps

### Case Studies
- [GitHub Copilot Technical Details](https://github.blog/2024-04-29-github-copilot-workspace/)
- [Intercom AI Architecture](https://www.intercom.com/blog/how-we-built-our-ai-agent/)
- [Stripe's ML Platform](https://stripe.com/blog/railyard-training-models) - Production ML

### Performance
- [The Tail at Scale](https://research.google/pubs/pub40801/) - Latency optimization
- [Caching Strategies](https://aws.amazon.com/caching/best-practices/) - AWS guide
- [Cost Optimization](https://www.anthropic.com/index/cost-effective-language-model-usage) - Anthropic strategies

### Reliability Patterns
- [Circuit Breaker Pattern](https://martinfowler.com/bliki/CircuitBreaker.html) - Martin Fowler
- [Release It!](https://pragprog.com/titles/mnee2/release-it-second-edition/) - Production patterns book
- [Chaos Engineering](https://principlesofchaos.org/) - Testing reliability

### Monitoring
- [Observability Engineering](https://www.oreilly.com/library/view/observability-engineering/9781492076438/) - O'Reilly book
- [Prometheus Best Practices](https://prometheus.io/docs/practices/) - Metrics collection
- [Distributed Tracing](https://opentelemetry.io/docs/) - OpenTelemetry guide